In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Ishan\OneDrive\Desktop\Projects\forest_cover_prediction\train.csv")
print(df.head())


   Id  Elevation  Aspect  Slope  Horizontal_Distance_To_Hydrology  \
0   1       2596      51      3                               258   
1   2       2590      56      2                               212   
2   3       2804     139      9                               268   
3   4       2785     155     18                               242   
4   5       2595      45      2                               153   

   Vertical_Distance_To_Hydrology  Horizontal_Distance_To_Roadways  \
0                               0                              510   
1                              -6                              390   
2                              65                             3180   
3                             118                             3090   
4                              -1                              391   

   Hillshade_9am  Hillshade_Noon  Hillshade_3pm  ...  Soil_Type32  \
0            221             232            148  ...            0   
1            220          

In [2]:
print(df.info())
print(df.describe())
print(df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15120 entries, 0 to 15119
Data columns (total 56 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   Id                                  15120 non-null  int64
 1   Elevation                           15120 non-null  int64
 2   Aspect                              15120 non-null  int64
 3   Slope                               15120 non-null  int64
 4   Horizontal_Distance_To_Hydrology    15120 non-null  int64
 5   Vertical_Distance_To_Hydrology      15120 non-null  int64
 6   Horizontal_Distance_To_Roadways     15120 non-null  int64
 7   Hillshade_9am                       15120 non-null  int64
 8   Hillshade_Noon                      15120 non-null  int64
 9   Hillshade_3pm                       15120 non-null  int64
 10  Horizontal_Distance_To_Fire_Points  15120 non-null  int64
 11  Wilderness_Area1                    15120 non-null  int64
 12  Wild

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['Elevation', 'Aspect', 'Slope']] = scaler.fit_transform(df[['Elevation', 'Aspect', 'Slope']])


In [32]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings("ignore")  # Ignore warnings for cleaner output

# Load dataset
file_path = r"C:\Users\Ishan\OneDrive\Desktop\Projects\forest_cover_prediction\train.csv"
df = pd.read_csv(file_path)

# Drop 'Id' as it's not useful for prediction
df.drop(columns=['Id'], inplace=True)

# Separate features and target
X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type']
y = y - 1  # Adjust labels to start from 0

# Identify numerical features (we assume all columns are used)
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Scale numerical features only
scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=35, stratify=y)

# **Pre-Saved Best Parameters (Assumed)**
best_params = {
    'n_estimators': 250,
    'max_depth': 15,
    'learning_rate': 0.65,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'tree_method': 'gpu_hist'
}

# Train final model with best parameters
print("\nTraining Final Model with Best Parameters...")
eval_set = [(X_train, y_train), (X_test, y_test)]

xgb_optimized = xgb.XGBClassifier(
    **best_params,
    eval_metric='mlogloss',
    random_state=60
)

xgb_optimized.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=10,  # Show training progress every 10 iterations
      # Stop if no improvement after 20 rounds
)

# Make predictions
y_pred = xgb_optimized.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOptimized XGBoost Model Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Training Final Model with Best Parameters...
[0]	validation_0-mlogloss:0.77179	validation_1-mlogloss:0.95711
[10]	validation_0-mlogloss:0.05400	validation_1-mlogloss:0.41053
[20]	validation_0-mlogloss:0.01883	validation_1-mlogloss:0.40099
[30]	validation_0-mlogloss:0.01138	validation_1-mlogloss:0.41013
[40]	validation_0-mlogloss:0.00822	validation_1-mlogloss:0.41782
[50]	validation_0-mlogloss:0.00662	validation_1-mlogloss:0.42800
[60]	validation_0-mlogloss:0.00555	validation_1-mlogloss:0.43271
[70]	validation_0-mlogloss:0.00486	validation_1-mlogloss:0.43774
[80]	validation_0-mlogloss:0.00435	validation_1-mlogloss:0.44355
[90]	validation_0-mlogloss:0.00396	validation_1-mlogloss:0.44805
[100]	validation_0-mlogloss:0.00365	validation_1-mlogloss:0.45192
[110]	validation_0-mlogloss:0.00340	validation_1-mlogloss:0.45624
[120]	validation_0-mlogloss:0.00320	validation_1-mlogloss:0.45876
[130]	validation_0-mlogloss:0.00302	validation_1-mlogloss:0.46239
[140]	validation_0-mlogloss:0.00286	valid